# Day 2: Data Cleaning & Preprocessing

This notebook implements a comprehensive data cleaning pipeline for the student scores dataset.

## 📋 Objectives
- Handle missing values appropriately
- Remove or investigate duplicate rows
- Detect and handle outliers
- Encode categorical variables
- Create clean dataset for modeling

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Load dataset
df = pd.read_csv('../../day4/student_scores.csv')
df_clean = df.copy()

print("📊 Original shape:", df.shape)
print("🔍 Missing values:\n", df.isnull().sum())
print("🔄 Duplicates:", df.duplicated().sum())

In [ ]:
# 1. Handle Missing Values
print("=== MISSING VALUE HANDLING ===")

# Check each column
for col in df_clean.columns:
    missing = df_clean[col].isnull().sum()
    if missing > 0:
        print(f"{col}: {missing} missing ({missing/len(df_clean)*100:.1f}%)")
        
        # Strategy based on data type
        if df_clean[col].dtype in ['float64', 'int64']:
            # Numerical: fill with median (robust to outliers)
            df_clean[col].fillna(df_clean[col].median(), inplace=True)
            print(f"  → Filled with median: {df_clean[col].median():.2f}")
        else:
            # Categorical: fill with mode
            df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)
            print(f"  → Filled with mode: {df_clean[col].mode()[0]}")
    else:
        print(f"{col}: ✅ No missing values")

In [ ]:
# 2. Handle Duplicates
print("=== DUPLICATE HANDLING ===")
dupes = df_clean.duplicated().sum()
print(f"Found {dupes} duplicate rows")

if dupes > 0:
    df_clean = df_clean.drop_duplicates()
    print(f"✅ Removed duplicates. New shape: {df_clean.shape}")
else:
    print("✅ No duplicates found")

In [ ]:
# 3. Outlier Detection using IQR Method
print("=== OUTLIER DETECTION (IQR Method) ===")

numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
outliers_summary = {}

for col in numeric_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df_clean[(df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)]
    outlier_count = len(outliers)
    outliers_summary[col] = outlier_count
    
    if outlier_count > 0:
        print(f"{col}: {outlier_count} outliers (bounds: {lower_bound:.2f} - {upper_bound:.2f})")
    else:
        print(f"{col}: ✅ No outliers")

In [ ]:
# 4. Visualize Outliers
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, col in enumerate(numeric_cols[:6]):
    sns.boxplot(data=df_clean, y=col, ax=axes[idx])
    axes[idx].set_title(f'Boxplot - {col}')

plt.tight_layout()
plt.savefig('../../day12/plots/outliers_boxplot.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 5. Categorical Encoding
print("=== CATEGORICAL ENCODING ===")

categorical_cols = df_clean.select_dtypes(include=['object']).columns
print(f"Categorical columns: {list(categorical_cols)}")

# One-hot encoding for nominal categories
df_encoded = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True)

print(f"\nOriginal shape: {df_clean.shape}")
print(f"Encoded shape: {df_encoded.shape}")
print(f"New columns: {list(df_encoded.columns)}")

In [ ]:
# 6. Feature Scaling (Standardization)
from sklearn.preprocessing import StandardScaler

print("=== FEATURE SCALING ===")

# Separate features and target (assuming last numeric column is target)
numeric_encoded = df_encoded.select_dtypes(include=[np.number])
target_col = numeric_encoded.columns[-1]  # Adjust as needed

X = numeric_encoded.drop(columns=[target_col])
y = numeric_encoded[target_col]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

print(f"Features shape: {X_scaled_df.shape}")
print(f"Target shape: {y.shape}")
print(f"Target column: {target_col}")

# Show scaled statistics
print("\nScaled features statistics:")
display(X_scaled_df.describe())

In [ ]:
# 7. Save Cleaned Dataset
print("=== SAVING CLEANED DATA ===")

# Save the fully cleaned and encoded dataset
df_encoded.to_csv('../../day12/data/student_scores_cleaned_encoded.csv', index=False)
print("✅ Saved: student_scores_cleaned_encoded.csv")

# Save scaled features separately
X_scaled_df.to_csv('../../day12/data/features_scaled.csv', index=False)
y.to_csv('../../day12/data/target.csv', index=False)
print("✅ Saved: features_scaled.csv & target.csv")

# Save scaler for later use
import joblib
joblib.dump(scaler, '../../day12/models/scaler.pkl')
print("✅ Saved: scaler.pkl")

## 📝 Summary

- Missing values handled with median/mode imputation
- Duplicates removed
- Outliers identified using IQR method
- Categorical variables one-hot encoded
- Features standardized using StandardScaler
- Clean datasets saved for modeling

---
*Next: [03_visualization.ipynb](03_visualization.ipynb)*